In [40]:
import pandas as pd
import numpy as np

In [41]:
df = pd.read_csv('dataset/anime.csv')

In [42]:
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


## Data Preprocessing:

**Load the dataset into a suitable data structure (e.g., pandas DataFrame).**

**Handle missing values, if any.**

**Explore the dataset to understand its structure and attributes.**


In [44]:
# Shape of dataset
df.shape

(12294, 7)

In [45]:
df.describe()

,anime_id,rating,members
count,12294.000000,12064.000000,1.229400e+04
mean,14058.221653,6.473902,1.807134e+04
std,11455.294701,1.026746,5.482068e+04
min,1.000000,1.670000,5.000000e+00
25%,3484.250000,5.880000,2.250000e+02
50%,10260.500000,6.570000,1.550000e+03
75%,24794.500000,7.180000,9.437000e+03
max,34527.000000,10.000000,1.013917e+06


In [46]:
# Checking Null values

df.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [47]:
# Drop rows where genre or rating is missing

df = df.dropna(subset=['genre','rating'])

In [48]:
# Fill missing type with 'Unknown'

df['type'] = df['type'].fillna('Unknown')

In [49]:
df.dtypes

anime_id      int64
name         object
genre        object
type         object
episodes     object
rating      float64
members       int64
dtype: object

In [50]:
df['episodes']

0         1
1        64
2        51
3        24
4        51
         ..
12289     1
12290     1
12291     4
12292     1
12293     1
Name: episodes, Length: 12017, dtype: object

In [51]:
# Convert episodes to numeric

df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')

In [52]:
df['episodes']

0         1.0
1        64.0
2        51.0
3        24.0
4        51.0
         ... 
12289     1.0
12290     1.0
12291     4.0
12292     1.0
12293     1.0
Name: episodes, Length: 12017, dtype: float64

In [53]:
# Replace missing episode values with median

df['episodes'].fillna(df['episodes'].median(), inplace=True)

In [54]:
print("Missing values after cleaning:")
print(df.isnull().sum())

Missing values after cleaning:
anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64


Here, the Anime dataset was successfully loaded into a Pandas DataFrame. Missing values were identified and handled appropriately. The dataset was explored to understand its structure, attributes, and potential issues. After preprocessing, the dataset is clean and suitable for building a cosine similarity-based recommendation system

## Feature Extraction:

**Decide on the features that will be used for computing similarity (e.g., genres, user ratings).**

**Convert categorical features into numerical representations if necessary.**

**Normalize numerical features if required.**

In [55]:
df.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1.0,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64.0,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51.0,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24.0,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51.0,9.16,151266


We will:

* Apply TF-IDF Vectorization on text features (genre, type)

* Normalize numerical features (episodes, rating)

* Combine all features into a single feature matrix

In [56]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Combine genre and type into one feature 
df['combined_features'] = df['genre'] + ' ' + df['type']

In [57]:
df['combined_features']

0               Drama, Romance, School, Supernatural Movie
1        Action, Adventure, Drama, Fantasy, Magic, Mili...
2        Action, Comedy, Historical, Parody, Samurai, S...
3                                      Sci-Fi, Thriller TV
4        Action, Comedy, Historical, Parody, Samurai, S...
                               ...                        
12289                                           Hentai OVA
12290                                           Hentai OVA
12291                                           Hentai OVA
12292                                           Hentai OVA
12293                                         Hentai Movie
Name: combined_features, Length: 12017, dtype: object

In [58]:
tfidf = TfidfVectorizer(stop_words = 'english')

# Transform text data into TF-IDf matrix
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

print('TF-IDF Matrix Shape: ', tfidf_matrix.shape)

TF-IDF Matrix Shape:  (12017, 51)


In [60]:
# Normalizing Numerical Features

from sklearn.preprocessing import MinMaxScaler

# Num Features
num_features = df[['episodes', 'rating']]

sc = MinMaxScaler()

num_scaled = sc.fit_transform(num_features)

In [62]:
# Combining Text and Numerical Features

from scipy.sparse import hstack
# hstack() combines sparse and dense matrices.

feature_matrix = hstack([tfidf_matrix, num_scaled])

print("Final Feature Matrix Shape:", feature_matrix.shape)

Final Feature Matrix Shape: (12017, 53)


In this task, relevant features for the recommendation system were selected and transformed into numerical form. Textual features were converted using TF-IDF vectorization, and numerical features were normalized using MinMaxScaler. All features were combined into a single feature matrix

## Recommendation System:

**Design a function to recommend anime based on cosine similarity.**

**Given a target anime, recommend a list of similar anime based on cosine similarity scores.**

**Experiment with different threshold values for similarity scores to adjust the recommendation list size.**

**Analyze the performance of the recommendation system and identify areas of improvement.**


In [63]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(feature_matrix, feature_matrix)

print('Cosine Similarity Matrix Shape: ', cosine_sim.shape)

Cosine Similarity Matrix Shape:  (12017, 12017)


In [64]:
df.head()

,anime_id,name,genre,type,episodes,rating,members,combined_features
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1.0,9.37,200630,"Drama, Romance, School, Supernatural Movie"
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64.0,9.26,793665,"Action, Adventure, Drama, Fantasy, Magic, Mili..."
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51.0,9.25,114262,"Action, Comedy, Historical, Parody, Samurai, S..."
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24.0,9.17,673572,"Sci-Fi, Thriller TV"
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51.0,9.16,151266,"Action, Comedy, Historical, Parody, Samurai, S..."


In [65]:
# Reset index for mapping
df = df.reset_index()

In [70]:
df.head()

,index,anime_id,name,genre,type,episodes,rating,members,combined_features
0,0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1.0,9.37,200630,"Drama, Romance, School, Supernatural Movie"
1,1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64.0,9.26,793665,"Action, Adventure, Drama, Fantasy, Magic, Mili..."
2,2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51.0,9.25,114262,"Action, Comedy, Historical, Parody, Samurai, S..."
3,3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24.0,9.17,673572,"Sci-Fi, Thriller TV"
4,4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51.0,9.16,151266,"Action, Comedy, Historical, Parody, Samurai, S..."


In [67]:
# Create index mapping
indices = pd.Series(df.index, index=df['name']).drop_duplicates()

In [69]:
print(indices)

name
Kimi no Na wa.                                            0
Fullmetal Alchemist: Brotherhood                          1
Gintama°                                                  2
Steins;Gate                                               3
Gintama&#039;                                             4
                                                      ...  
Toushindai My Lover: Minami tai Mecha-Minami          12012
Under World                                           12013
Violence Gekiga David no Hoshi                        12014
Violence Gekiga Shin David no Hoshi: Inma Densetsu    12015
Yasuji no Pornorama: Yacchimae!!                      12016
Length: 12017, dtype: int64


In [71]:
def recommend_anime(title, top_n=10, threshold=0.3):
    if title not in indices:
        return "Anime not found in dataset."
    
    idx = indices[title]
    
    # Get similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort based on similarity score
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Remove itself (first item)
    sim_scores = sim_scores[1:]
    
    # Apply threshold
    sim_scores = [i for i in sim_scores if i[1] >= threshold]
    
    # Get top recommendations
    sim_scores = sim_scores[:top_n]
    
    anime_indices = [i[0] for i in sim_scores]
    
    return df[['name', 'genre', 'rating']].iloc[anime_indices]

In [81]:
print(indices)

name
Kimi no Na wa.                                            0
Fullmetal Alchemist: Brotherhood                          1
Gintama°                                                  2
Steins;Gate                                               3
Gintama&#039;                                             4
                                                      ...  
Toushindai My Lover: Minami tai Mecha-Minami          12012
Under World                                           12013
Violence Gekiga David no Hoshi                        12014
Violence Gekiga Shin David no Hoshi: Inma Densetsu    12015
Yasuji no Pornorama: Yacchimae!!                      12016
Length: 12017, dtype: int64


In [87]:
# Generate Recommendation

recommend_anime("Steins;Gate", top_n=10, threshold=0.4)

,name,genre,rating
59,Steins;Gate Movie: Fuka Ryouiki no Déjà vu,"Sci-Fi, Thriller",8.61
126,Steins;Gate: Oukoubakko no Poriomania,"Sci-Fi, Thriller",8.46
196,Steins;Gate: Kyoukaimenjou no Missing Link - D...,"Sci-Fi, Thriller",8.34
5124,Under the Dog,"Action, Sci-Fi, Thriller",6.55
238,Gankutsuou,"Drama, Mystery, Sci-Fi, Supernatural, Thriller",8.27
493,Higashi no Eden,"Action, Comedy, Drama, Mystery, Romance, Sci-F...",8.03
2518,Ibara no Ou,"Action, Mystery, Sci-Fi, Thriller",7.24
5523,Loups=Garous,"Mystery, Sci-Fi, Thriller",6.43
6885,Loups=Garous Pilot,"Mystery, Sci-Fi, Thriller",5.87
36,Fate/Zero 2nd Season,"Action, Fantasy, Supernatural, Thriller",8.73


In [85]:
# Experiment with Different Threshold Values

print("Threshold = 0.2")
display(recommend_anime("Steins;Gate", top_n=10, threshold=0.2))

print("Threshold = 0.4")
display(recommend_anime("Steins;Gate", top_n=10, threshold=0.4))

print("Threshold = 0.6")
display(recommend_anime("Steins;Gate", top_n=10, threshold=0.6))

Threshold = 0.2


,name,genre,rating
59,Steins;Gate Movie: Fuka Ryouiki no Déjà vu,"Sci-Fi, Thriller",8.61
126,Steins;Gate: Oukoubakko no Poriomania,"Sci-Fi, Thriller",8.46
196,Steins;Gate: Kyoukaimenjou no Missing Link - D...,"Sci-Fi, Thriller",8.34
5124,Under the Dog,"Action, Sci-Fi, Thriller",6.55
238,Gankutsuou,"Drama, Mystery, Sci-Fi, Supernatural, Thriller",8.27
493,Higashi no Eden,"Action, Comedy, Drama, Mystery, Romance, Sci-F...",8.03
2518,Ibara no Ou,"Action, Mystery, Sci-Fi, Thriller",7.24
5523,Loups=Garous,"Mystery, Sci-Fi, Thriller",6.43
6885,Loups=Garous Pilot,"Mystery, Sci-Fi, Thriller",5.87
36,Fate/Zero 2nd Season,"Action, Fantasy, Supernatural, Thriller",8.73


Threshold = 0.4


,name,genre,rating
59,Steins;Gate Movie: Fuka Ryouiki no Déjà vu,"Sci-Fi, Thriller",8.61
126,Steins;Gate: Oukoubakko no Poriomania,"Sci-Fi, Thriller",8.46
196,Steins;Gate: Kyoukaimenjou no Missing Link - D...,"Sci-Fi, Thriller",8.34
5124,Under the Dog,"Action, Sci-Fi, Thriller",6.55
238,Gankutsuou,"Drama, Mystery, Sci-Fi, Supernatural, Thriller",8.27
493,Higashi no Eden,"Action, Comedy, Drama, Mystery, Romance, Sci-F...",8.03
2518,Ibara no Ou,"Action, Mystery, Sci-Fi, Thriller",7.24
5523,Loups=Garous,"Mystery, Sci-Fi, Thriller",6.43
6885,Loups=Garous Pilot,"Mystery, Sci-Fi, Thriller",5.87
36,Fate/Zero 2nd Season,"Action, Fantasy, Supernatural, Thriller",8.73


Threshold = 0.6


,name,genre,rating
59,Steins;Gate Movie: Fuka Ryouiki no Déjà vu,"Sci-Fi, Thriller",8.61
126,Steins;Gate: Oukoubakko no Poriomania,"Sci-Fi, Thriller",8.46
196,Steins;Gate: Kyoukaimenjou no Missing Link - D...,"Sci-Fi, Thriller",8.34
5124,Under the Dog,"Action, Sci-Fi, Thriller",6.55
238,Gankutsuou,"Drama, Mystery, Sci-Fi, Supernatural, Thriller",8.27
493,Higashi no Eden,"Action, Comedy, Drama, Mystery, Romance, Sci-F...",8.03
2518,Ibara no Ou,"Action, Mystery, Sci-Fi, Thriller",7.24
5523,Loups=Garous,"Mystery, Sci-Fi, Thriller",6.43
6885,Loups=Garous Pilot,"Mystery, Sci-Fi, Thriller",5.87
36,Fate/Zero 2nd Season,"Action, Fantasy, Supernatural, Thriller",8.73


The recommendation results were generated for the anime “Steins;Gate” using different similarity thresholds.The recommendation list remains almost identical for thresholds 0.2, 0.4, and 0.6.

**Key Observations:**

1. Most recommended anime belong to similar genres:

* Sci-Fi

* Thriller

* Mystery

* Supernatural

2. Direct franchise-related recommendations appear at the top:

* Steins;Gate Movie

* Steins;Gate Specials

3. Closely related genre-based anime such as:

* Gankutsuou

* Higashi no Eden

* Ibara no Ou

4. Even at a high threshold (0.6), recommendations remain unchanged.

**Limitations**
1. Threshold Not Impacting Output Significantly

Since results remain the same across thresholds, this suggests:

* Similarity scores are clustered in a narrow high range.

* Numerical features (episodes, rating) may have limited influence.

* Genre similarity dominates the system.

2. Over-Reliance on Genre

The system mainly relies on textual similarity from genre.

## Areas of Improvement:

**Apply Feature Weighting**

Currently, all features contribute equally. Improve by:

* Assigning higher weight to genre

* Moderate weight to rating

* Lower weight to episodes

**Dimensionality Reduction**

Use PCA to reduce noise and improve computational efficiency.

**Use Advanced Embeddings**

Instead of TF-IDF, use:

* Word2Vec

* FastText

* Transformer embeddings

## Interview Questions:

**1. Can you explain the difference between user-based and item-based collaborative filtering?**

Collaborative Filtering is a recommendation technique that suggests items based on user interactions and preferences. It is mainly divided into:

* User-Based Collaborative Filtering

* Item-Based Collaborative Filtering

**User-Based Collaborative Filtering** 

User-based collaborative filtering recommends items to a user based on the preferences of other users who have similar tastes.

If:

User A and User B both liked C and D

User B also liked E

Then:

E will be recommended to User A.

**Item-Based Collaborative Filtering**

Item-based collaborative filtering recommends items based on similarity between items rather than users.

If:

Many users who liked A also liked B

Then:

B will be recommended to users who liked A.

| Feature                     | User-Based CF      | Item-Based CF       |
| --------------------------- | ------------------ | ------------------- |
| Similarity Computed Between | Users              | Items               |
| Focus                       | User behavior      | Item similarity     |
| Scalability                 | Less scalable      | More scalable       |
| Stability                   | Changes frequently | More stable         |
| Used In                     | Small systems      | Large-scale systems |


**2. What is collaborative filtering, and how does it work?**

Collaborative Filtering (CF) is a recommendation technique that predicts a user’s interests by analyzing preferences and behaviors of many users.

It works on the principle:

Users who liked similar items in the past will likely like similar items in the future.

Unlike content-based filtering, collaborative filtering does not rely on item features (like genre), it relies on user interaction data such as ratings, purchases, clicks, or watch history.

**Collaborative Filtering Working**

It works in three main steps:

**1. Build User-Item Interaction Matrix**

Create a matrix where:

* Rows → Users

* Columns → Items

* Values → Ratings / Interactions

**2️. Compute Similarity**

Similarity can be calculated using:

* Cosine Similarity

* Pearson Correlation

* Euclidean Distance

Similarity can be computed between:

* Users (User-based CF)

* Items (Item-based CF)

**3️. Generate Recommendations**

Based on similarity:

* Recommend items liked by similar users (User-based)

* Recommend items similar to items user already liked (Item-based)